# Etapa 3 · Fidelidad del CVAE + demo de power

Dos preguntas:

1. **Fidelidad distribucional**: que tan cerca estan las ventanas generadas por el CVAE de las reales? Combinacion de tests marginales (Mann-Whitney U) y un *discriminative score* (Esteban et al., 2017).
2. **Power**: cuanto sube la power de un ensayo clinico de muestra pequena al complementar datos reales con sinteticos, controlando el error Tipo-I?

Dependencias: `tremor_X_*` (Etapa 1) + `cvae_best.pt`, `synth_X.npy` (Etapa 2).

## 0. Carga

In [1]:
import numpy as np, json, os
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sp_signal, stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

np.random.seed(42)

X_train = np.load('tremor_X_train.npy'); y_train = np.load('tremor_y_train.npy')
X_test  = np.load('tremor_X_test.npy');  y_test  = np.load('tremor_y_test.npy')
synth_X = np.load('synth_X.npy');        synth_y = np.load('synth_y.npy')

with open('label_dict.json') as f:
    LABELS = json.load(f)
COND_INV = {v: k for k, v in LABELS['condition'].items()}
WINDOW_LEN = LABELS['window_len']; FS = LABELS['sampling_rate']

print(f'Train real {X_train.shape}, Test real {X_test.shape}, Synth {synth_X.shape}')

Train real (7485, 240), Test real (1899, 240), Synth (4000, 240)


## 1. Features por ventana

- `peak_freq_hz`, `tremor_band_ratio` (4-6 Hz / total)
- `log_band_power` -- endpoint principal del LME
- `rms`, `spectral_entropy`, `kurtosis`

In [2]:
def features_window(x, fs):
    f, p = sp_signal.welch(x, fs=fs, nperseg=len(x))
    band = (f >= 4) & (f <= 6)
    band_power = p[band].sum()
    total = p.sum() + 1e-12
    return {
        'peak_freq_hz':      float(f[np.argmax(p)]),
        'tremor_band_ratio': float(band_power / total),
        'log_band_power':    float(np.log(band_power + 1e-12)),
        'rms':               float(np.sqrt(np.mean(x**2))),
        'spectral_entropy':  float(-np.sum((p / total) * np.log(p / total + 1e-12))),
        'kurtosis':          float(stats.kurtosis(x)),
    }

def features_batch(X, fs):
    return pd.DataFrame([features_window(x, fs) for x in X])

feat_real_train = features_batch(X_train, FS); feat_real_train['cond'] = y_train[:, 0]
feat_real_test  = features_batch(X_test,  FS); feat_real_test['cond']  = y_test[:, 0]
feat_synth      = features_batch(synth_X, FS); feat_synth['cond']      = synth_y
print('shapes:', feat_real_train.shape, feat_real_test.shape, feat_synth.shape)

shapes: (7485, 7) (1899, 7) (4000, 7)


## 2. Test marginal de dos muestras

Mann-Whitney U por (condicion x feature) entre real y sintetico, con correccion Benjamini-Hochberg. Se desea p_BH > 0.05 (distribuciones indistinguibles marginal-mente).

In [3]:
feat_cols = ['peak_freq_hz', 'tremor_band_ratio', 'log_band_power', 'rms', 'spectral_entropy', 'kurtosis']
records = []
for cond_idx in sorted(LABELS['condition'].values()):
    sub_real  = feat_real_train[feat_real_train['cond'] == cond_idx]
    sub_synth = feat_synth[feat_synth['cond'] == cond_idx]
    for col in feat_cols:
        if len(sub_real) == 0 or len(sub_synth) == 0:
            continue
        u, p = stats.mannwhitneyu(sub_real[col], sub_synth[col], alternative='two-sided')
        records.append({
            'condition':  COND_INV[cond_idx],
            'feature':    col,
            'real_mean':  sub_real[col].mean(),
            'real_std':   sub_real[col].std(),
            'synth_mean': sub_synth[col].mean(),
            'synth_std':  sub_synth[col].std(),
            'p_raw':      p,
        })
df_mw = pd.DataFrame(records)
df_mw['p_bh'] = multipletests(df_mw['p_raw'], method='fdr_bh')[1]
df_mw['significant'] = df_mw['p_bh'] < 0.05
print(df_mw.round(4).to_string(index=False))
df_mw.to_csv('fidelity_mw.csv', index=False)

      condition           feature  real_mean  real_std  synth_mean  synth_std  p_raw   p_bh  significant
DBS_OFF_MED_OFF      peak_freq_hz     6.2594    2.3766      5.6650     1.4559 0.0000 0.0000         True
DBS_OFF_MED_OFF tremor_band_ratio     0.4049    0.3038      0.4588     0.1858 0.0000 0.0000         True
DBS_OFF_MED_OFF    log_band_power    -1.6258    1.6333     -1.8663     0.9760 0.0000 0.0000         True
DBS_OFF_MED_OFF               rms     0.9034    0.5689      0.6187     0.1979 0.0000 0.0000         True
DBS_OFF_MED_OFF  spectral_entropy     2.1670    0.6059      2.0301     0.2158 0.0000 0.0000         True
DBS_OFF_MED_OFF          kurtosis     0.8849    3.9394      0.3863     1.3614 0.0000 0.0000         True
 DBS_OFF_MED_ON      peak_freq_hz     6.3175    2.7957      5.7350     1.5045 0.0000 0.0000         True
 DBS_OFF_MED_ON tremor_band_ratio     0.2441    0.1949      0.4302     0.1827 0.0000 0.0000         True
 DBS_OFF_MED_ON    log_band_power    -3.8423    2.7060 

## 3. Discriminative score (Esteban et al., 2017)

Se entrena una LR para distinguir real vs sintetico sobre las 6 features, con CV 5-fold estratificado. AUROC ~0.5 => indistinguibles; >0.7 => diferencias notables.

(Nota: este test no es TSTR -- en TSTR el clasificador se entrena en sinteticos y se evalua en reales para una tarea downstream. Aqui medimos *distinguibilidad*.)

In [4]:
X_clf = pd.concat([feat_real_train[feat_cols], feat_synth[feat_cols]], ignore_index=True).values
y_clf = np.concatenate([np.zeros(len(feat_real_train)), np.ones(len(feat_synth))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aurocs = []
for tr, te in skf.split(X_clf, y_clf):
    clf = LogisticRegression(max_iter=500, C=1.0).fit(X_clf[tr], y_clf[tr])
    aurocs.append(roc_auc_score(y_clf[te], clf.predict_proba(X_clf[te])[:, 1]))
print(f'discriminative AUROC = {np.mean(aurocs):.3f} +- {np.std(aurocs):.3f}')
print('  (cerca de 0.5 = synth indistinguible de real; >0.7 = diferencias significativas)')

discriminative AUROC = 0.808 +- 0.006
  (cerca de 0.5 = synth indistinguible de real; >0.7 = diferencias significativas)


## 4. Demo de power

- Endpoint: $Y = \log(\text{band power 4-6 Hz})$
- Diseno factorial 2x2 (DBS x MED), crossover dentro de sujeto
- LME: `Y ~ DBS + MED + DBS:MED + (1|subject)`, Wald test sobre el efecto principal DBS, alfa=0.05

Grid: `n_real in {4, 8, 12, 16}`, `m_synth in {0, 16, 64, 256}` por condicion. 200 simulaciones Monte Carlo por celda.

**Caveat sobre la estructura "sujeto" de los datos sinteticos**: cada ventana sintetica se asigna a un sujeto virtual de forma que cada sujeto virtual contribuye una ventana por cada una de las 4 condiciones (estructura crossover balanceada). Pero como los z latentes se muestrean i.i.d. desde N(0, I), no hay correlacion intra-sujeto real, por lo que la varianza del intercepto aleatorio para sujetos virtuales tiende a 0. Es una limitacion conocida; por eso monitoreamos Tipo-I al mismo tiempo que power.

In [5]:
def cond_to_dbs_med(cond_idx, label_map):
    name = {v: k for k, v in label_map.items()}[cond_idx]
    return (1 if 'DBS_ON' in name else 0,
            1 if 'MED_ON' in name else 0)

# Real: subject_num real (de y_train[:, 3]).
real_df = feat_real_train.copy()
real_df['dbs'], real_df['med'] = zip(*[cond_to_dbs_med(c, LABELS['condition']) for c in real_df['cond']])
real_df['subj_num'] = y_train[:, 3]

# Sintetico: asignacion crossover (una ventana por cada condicion por sujeto virtual).
synth_df = feat_synth.copy()
synth_df['dbs'], synth_df['med'] = zip(*[cond_to_dbs_med(c, LABELS['condition']) for c in synth_df['cond']])
synth_df = synth_df.sort_values('cond').reset_index(drop=True)
synth_df['subj_num'] = -1
for c, sub in synth_df.groupby('cond'):
    # Mismos ids virtuales en cada condicion -> el sujeto k tiene una ventana en cada condicion.
    synth_df.loc[sub.index, 'subj_num'] = np.arange(len(sub)) + 1000

print('real:', real_df.shape, ' synth:', synth_df.shape)

real: (7485, 10)  synth: (4000, 10)


In [ ]:
def fit_lme_pvalue(df):
    try:
        md = smf.mixedlm('log_band_power ~ dbs + med + dbs:med', df, groups=df['subj_num'])
        res = md.fit(reml=False, method='lbfgs', disp=False)
        return float(res.pvalues['dbs'])
    except Exception:
        return float('nan')


def one_simulation(n_real, m_synth_per_cond, real_df, synth_df, permute, rng):
    real_subjs = real_df['subj_num'].unique()
    chosen = rng.choice(real_subjs, size=min(n_real, len(real_subjs)), replace=False)
    sub_real = real_df[real_df['subj_num'].isin(chosen)]

    sub_synth_list = []
    if m_synth_per_cond > 0:
        for c in synth_df['cond'].unique():
            pool = synth_df[synth_df['cond'] == c]
            n_take = min(m_synth_per_cond, len(pool))
            sub_synth_list.append(pool.sample(n=n_take, random_state=rng.randint(0, 10**9)))
    sub_synth = pd.concat(sub_synth_list, ignore_index=True) if sub_synth_list else pd.DataFrame()
    df_all = pd.concat([sub_real, sub_synth], ignore_index=True)

    if permute:
        df_all = df_all.copy()
        df_all['dbs'] = rng.permutation(df_all['dbs'].values)
    return fit_lme_pvalue(df_all)


N_GRID = [4, 8, 12, 16]
M_GRID = [0, 16, 64, 256]
N_MC   = 200

power_grid = np.zeros((len(N_GRID), len(M_GRID)))
type1_grid = np.zeros((len(N_GRID), len(M_GRID)))

rng = np.random.RandomState(2026)
for i, n in enumerate(N_GRID):
    for j, m in enumerate(M_GRID):
        p_alt, p_null = [], []
        for _ in range(N_MC):
            p_alt.append( one_simulation(n, m, real_df, synth_df, permute=False, rng=rng))
            p_null.append(one_simulation(n, m, real_df, synth_df, permute=True,  rng=rng))
        p_alt  = np.array(p_alt);  p_alt  = p_alt[~np.isnan(p_alt)]
        p_null = np.array(p_null); p_null = p_null[~np.isnan(p_null)]
        power_grid[i, j] = (p_alt  < 0.05).mean() if len(p_alt)  else np.nan
        type1_grid[i, j] = (p_null < 0.05).mean() if len(p_null) else np.nan
        print(f'n={n:2d} m={m:3d}: power={power_grid[i,j]:.3f}  type1={type1_grid[i,j]:.3f}')

np.save('power_grid.npy', power_grid)
np.save('type1_grid.npy', type1_grid)

/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_line

n= 4 m=  0: power=0.890  type1=0.030


/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_line

n= 4 m= 16: power=0.825  type1=0.045


/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserW

n= 4 m= 64: power=0.870  type1=0.060


/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/Users/echozhang/opt/anaconda3/envs/cvae-tremor/lib/python3.11/site-packages/statsmodels/regression/mixed_line

## 5. Superficie power x Tipo-I

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, grid, title, vmin, vmax, cmap in [
    (axes[0], power_grid, 'Empirical Power', 0, 1, 'viridis'),
    (axes[1], type1_grid, 'Empirical Type-I (target <= 0.05)', 0, 0.2, 'magma'),
]:
    im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(M_GRID))); ax.set_xticklabels([str(m) for m in M_GRID])
    ax.set_yticks(range(len(N_GRID))); ax.set_yticklabels([str(n) for n in N_GRID])
    ax.set_xlabel('m (ventanas sinteticas / condicion)')
    ax.set_ylabel('n (sujetos reales)')
    ax.set_title(title)
    for i in range(len(N_GRID)):
        for j in range(len(M_GRID)):
            ax.text(j, i, f'{grid[i,j]:.2f}', ha='center', va='center',
                    color='white' if grid[i,j] < (vmin+vmax)/2 else 'black', fontsize=9)
    plt.colorbar(im, ax=ax)

plt.suptitle('Aumento de datos sinteticos con CVAE - power y Tipo-I')
plt.tight_layout(); plt.savefig('power_grid.png', dpi=130); plt.show()

## Resumen

| Item | Resultado |
|---|---|
| Mann-Whitney U (BH-adj) | ver `fidelity_mw.csv` |
| Discriminative AUROC | cerca de 0.5 => synth indistinguible; >0.7 => requiere ajuste |
| Mejora de power @ n=4 | diferencia entre m=0 y m=64 |
| Tipo-I (target <= 0.05) | limite superior aceptable ~0.075 (Thorlund et al., 2020) |

Figuras para el informe: `psd_real_vs_synth.png`, `recon_check.png`, `power_grid.png`. Tabla principal: `fidelity_mw.csv`.